# Bias Mitigation in Translation — training & evaluation

This notebook fine-tunes a translation LLM to mitigate **gender bias** and
deploys the resulting artifact to the `Fine-tuned model endpoint (unbiased)`
used by the live Translation service.

Steps:
1. load a toy translation dataset with gendered source sentences;
2. compute a simple gender-bias metric over the current (biased) model's
   translations;
3. "fine-tune" the model to reduce gender bias by learning a debiasing
   glossary from the dataset;
4. re-evaluate the bias metric after fine-tuning;
5. upload the fine-tuned model artifact (the debiasing glossary) to the
   `model-artifacts` S3 bucket, from where the `Fine-tuned model endpoint`
   Lambda loads it at inference time — i.e. "deploying" the model.

In [ ]:
import json

import boto3

FLOCI_ENDPOINT_URL = "http://localhost:4566"
AWS_REGION = "us-east-1"
MODEL_ARTIFACTS_BUCKET = "floci-translation-bias-mitigation-model-artifacts"
DEBIAS_GLOSSARY_KEY = "fine-tuned-translation-model/debias_glossary.json"


## 1. Load a toy translation dataset

A handful of English sentences containing gendered occupational terms and
pronouns, paired with the translations a naive (biased) model currently
produces.

In [ ]:
dataset = [
    {
        "source": "The chairman said he would review it.",
        "biased_translation": "El chairman dijo que he would review it.",
    },
    {
        "source": "The waitress told her manager about the issue.",
        "biased_translation": "La waitress le dijo a her manager sobre el problema.",
    },
    {
        "source": "Every policeman must protect mankind.",
        "biased_translation": "Cada policeman debe proteger a mankind.",
    },
    {
        "source": "She said the chairman would call back.",
        "biased_translation": "She dijo que el chairman volveria a llamar.",
    },
]

for row in dataset:
    print(row["source"])


## 2. Compute a gender-bias metric

A simplified metric: count how many gendered / non-neutral terms (from a
known list of biased terms) appear in the model's translations. A lower
score means a less gender-biased model.

In [ ]:
BIASED_TERMS = ["chairman", "waitress", "policeman", "mankind", "he said", "she said", " he ", " she ", "her manager"]


def gender_bias_score(translations):
    """Counts occurrences of known biased terms across a set of translations."""
    total = 0
    for text in translations:
        lowered = text.lower()
        total += sum(lowered.count(term) for term in BIASED_TERMS)
    return total


baseline_translations = [row["biased_translation"] for row in dataset]
baseline_score = gender_bias_score(baseline_translations)
print(f"Baseline gender-bias score (pre fine-tuning): {baseline_score}")


## 3. Fine-tune: learn a debiasing glossary

Instead of a full LLM fine-tuning run (out of scope for this local scaffold),
we simulate fine-tuning by learning a glossary that maps biased terms to
gender-neutral equivalents. This glossary **is** the fine-tuned model
artifact deployed to the `Fine-tuned model endpoint (unbiased)` Lambda.

In [ ]:
debias_glossary = {
    "chairman": "chairperson",
    "waitress": "waiter",
    "policeman": "police officer",
    "mankind": "humankind",
    "he said": "they said",
    "she said": "they said",
}


def apply_debias_glossary(text, glossary):
    debiased = text
    for biased_term, neutral_term in glossary.items():
        if biased_term in debiased:
            debiased = debiased.replace(biased_term, neutral_term)
    return debiased


fine_tuned_translations = [
    apply_debias_glossary(row["biased_translation"], debias_glossary) for row in dataset
]

for before, after in zip(baseline_translations, fine_tuned_translations):
    print(f"- {before}\n  -> {after}")


## 4. Re-evaluate the gender-bias metric

Confirm the fine-tuned model reduces the gender-bias score versus the
baseline.

In [ ]:
fine_tuned_score = gender_bias_score(fine_tuned_translations)
print(f"Fine-tuned gender-bias score (post fine-tuning): {fine_tuned_score}")

reduction_pct = 100 * (baseline_score - fine_tuned_score) / baseline_score
print(f"Bias reduction: {reduction_pct:.1f}%")

assert fine_tuned_score < baseline_score, "Fine-tuning should reduce the gender-bias score"


## 5. Upload the fine-tuned artifact to S3 (deploy the model)

Upload the debiasing glossary to the `model-artifacts` bucket. The
`Fine-tuned model endpoint (unbiased)` Lambda reads this object at inference
time, so uploading a new glossary here effectively **deploys** the
newly fine-tuned, unbiased model to the live translation service — no Lambda
redeploy required.

In [ ]:
s3 = boto3.client("s3", region_name=AWS_REGION, endpoint_url=FLOCI_ENDPOINT_URL)

s3.put_object(
    Bucket=MODEL_ARTIFACTS_BUCKET,
    Key=DEBIAS_GLOSSARY_KEY,
    Body=json.dumps(debias_glossary).encode("utf-8"),
    ContentType="application/json",
)

print(f"Uploaded fine-tuned, unbiased model artifact to s3://{MODEL_ARTIFACTS_BUCKET}/{DEBIAS_GLOSSARY_KEY}")


## Next steps

With the `floci` emulator and Terraform stack running (`docker compose up -d`
and `terraform apply`), re-running this notebook end-to-end retrains and
redeploys the gender-debiased translation model used by the
`update-endpoint` Lambda and the Translation service web app.